# Flows · **10 — Gold: what comes off the waiting list, and how it is counted**

**The question.** The waiting list is a stock. Each month it should change by

> clock starts − reported completed pathways − derived unreported completed pathways and removals

NHS England defines the last term as the **balancing item** in that identity:
it is derived, not observed (RTT statistics user guidance, "Unreported
removals"; Overview time series note 12; the "Derived unreported removals"
explainer dated 14 April 2026, all saved in `data/bronze/nhse_evidence_flows_2026-11/`).

Before I use these flows for anything, this notebook checks that the identity
closes, month by month, April 2025 to the reporting month:

| gate | basis | what must hold |
|---|---|---|
| **F1** | headline (with NHS England's estimates for missing trusts) | start list + starts − completions − unreported = end list, every month; my derivation equals NHS England's published column |
| **F2** | headline | the end list equals the headline series the main gate (04) rebuilt, so the flows sit on the verified stock |
| **F3** | reported providers only | the same identity on the reported columns |
| **F4** | reported providers, provider by provider | provider clock starts and completions from the full CSV extracts sum exactly to NHS England's reported national flows |
| **F5** | reported providers | provider-level unreported removals sum exactly to the national reported figure |

It also explains why the headline flows are not whole numbers, sets a rounding
rule for quoting them, and says what provider-level unreported removals can
and can't mean. Nothing in notebooks 00-09 changes. This notebook reads their
outputs and the Bronze files they already use.

Outputs: `data/gold/10_flows/` and `data/silver/rtt/rtt_provider_flows.parquet`.

In [1]:
import sys
import zipfile
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import (BRONZE_DIR, FIRST_MONTH, GOLD_DIR, REPORT_MONTH,  # noqa: E402
                     RTT_UNIDENTIFIED_COMMISSIONER, SERIES_MONTHS, SILVER_DIR, TRUST_SUCCESSIONS,
                     bronze_file, mon_yy, month_year, shift_month)

OUT = GOLD_DIR / "10_flows"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
L = mon_yy(REPORT_MONTH)
checks = []

def check(gate, name, ok, note=""):
    checks.append({"gate": gate, "check": name, "pass": bool(ok), "note": note})

## Step 0 — The main gate must have passed for this reporting month

In [2]:
gate = pd.read_csv(GOLD_DIR / "00_reconciliation" / "reconciliation.csv")
assert gate["pass"].all(), "the main reconciliation gate (04) has failing checks: run 04 first"
assert gate.check.str.contains(L).any(), f"the main gate was not run for {month_year(REPORT_MONTH)}"
headline = pd.read_csv(GOLD_DIR / "00_reconciliation" / "headline_rebuilt.csv", index_col="period")
print(f"main gate: {len(gate)} of {len(gate)} checks passed for {month_year(REPORT_MONTH)}")

main gate: 52 of 52 checks passed for September 2026


## Step 1 — The flows, as NHS England publishes them

All from the Overview time series of the reporting month's release, located
by group header plus column header. Each measure comes in two versions: as
reported, and "with estimates for missing data". I need the month before the
window too, because the identity starts from the previous month's list.

In [3]:
ws = openpyxl.load_workbook(bronze_file(f"nhse_rtt_published_{REPORT_MONTH}/RTT-Overview-Timeseries-*-{L}-*.xlsx"),
                            read_only=True, data_only=True).worksheets[0]
orows = list(ws.iter_rows(values_only=True))
group, keys = None, []
for g_, h_ in zip(orows[10], orows[11]):
    group = str(g_).strip() if g_ else group
    keys.append(f"{group} | {str(h_).strip()}" if h_ else None)

COLS = {
    "list":       ("Incomplete RTT pathways | Total waiting (mil)", "Incomplete RTT pathways | Total waiting (mil) with estimates for missing data"),
    "admitted":   ("Completed admitted (unadjusted) RTT pathways | No. of pathways (all)",
                   "Completed admitted (unadjusted) RTT pathways | No. of pathways (all) with estimates for missing data"),
    "nonadmitted": ("Completed non-admitted RTT pathways | No. of pathways (all)",
                    "Completed non-admitted RTT pathways | No. of pathways (all) with estimates for missing data"),
    "starts":     ("New RTT periods | No. of new RTT periods", "New RTT periods | No. of new RTT periods with estimates for missing data"),
    "completed":  ("Total completed pathways | Total completed pathways", "Total completed pathways | Total completed pathways with estimates"),
    "unreported": ("Derived unreported completed pathways and removals | Total unreported completed pathways and removals",
                   "Derived unreported completed pathways and removals | Total unreported completed pathways and removals with estimates"),
    "removals":   ("Derived total removals | Derived total removals", "Derived total removals | Derived total removals with estimates"),
    "share":      ("Derived unreported completed pathways and removals as a % of total removals | "
                   "Derived unreported completed pathways and removals as a % of total removals",
                   "Derived unreported completed pathways and removals as a % of total removals | "
                   "Derived unreported completed pathways and removals as a % of total removals, with estimates"),
}
pos = {(m, b): keys.index(c) for m, (rep, est) in COLS.items() for b, c in (("reported", rep), ("headline", est))}
recs = []
for r in orows[12:]:
    if hasattr(r[2], "strftime"):
        recs.append({"period": r[2].strftime("%Y-%m"), **{f"{m}_{b}": r[i] for (m, b), i in pos.items()}})
ov = pd.DataFrame(recs).set_index("period").apply(pd.to_numeric, errors="coerce")
WINDOW = SERIES_MONTHS                                  # April 2025 to the reporting month
ov = ov.loc[shift_month(FIRST_MONTH, -1):REPORT_MONTH]
assert list(ov.index) == [shift_month(FIRST_MONTH, -1)] + WINDOW, "Overview is missing a month in the window"
assert ov.notna().all().all(), "a flow column is blank in the window"
ov.loc[WINDOW, [c for c in ov.columns if c.endswith("_headline")]].round(0).tail(4)

,list_headline,admitted_headline,nonadmitted_headline,starts_headline,completed_headline,unreported_headline,removals_headline,share_headline
period,,,,,,,,
2026-06,"7,272,291.00","324,247.00","1,332,290.00","1,968,600.00","1,656,538.00","318,155.00","1,974,693.00",0.00
2026-07,"7,328,252.00","328,929.00","1,322,549.00","1,985,434.00","1,651,478.00","277,994.00","1,929,473.00",0.00
2026-08,"7,370,100.00","292,411.00","1,138,396.00","1,717,335.00","1,430,808.00","244,679.00","1,675,487.00",0.00
2026-09,"7,375,866.00","323,789.00","1,349,881.00","1,981,844.00","1,673,669.00","302,409.00","1,976,078.00",0.00


## Step 2 — Gate F1/F2: does the identity close on the headline basis?

For each month *t*:

> derived unreported(t) = list(t−1) + starts(t) − completed(t) − list(t)

and it must equal NHS England's published "with estimates" column. I also check
NHS England's own sub-totals (completed = admitted + non-admitted; total removals
= completed + unreported; share = unreported ÷ total removals), and that the
end-of-month list is the headline the main gate rebuilt (F2).

In [4]:
def identity(b: str) -> pd.DataFrame:
    o = ov
    d = pd.DataFrame(index=WINDOW)
    d["list_start"] = o[f"list_{b}"].shift(1).loc[WINDOW]
    d["starts"] = o.loc[WINDOW, f"starts_{b}"]
    d["admitted"] = o.loc[WINDOW, f"admitted_{b}"]
    d["nonadmitted"] = o.loc[WINDOW, f"nonadmitted_{b}"]
    d["completed"] = o.loc[WINDOW, f"completed_{b}"]
    d["unreported_published"] = o.loc[WINDOW, f"unreported_{b}"]
    d["list_end"] = o.loc[WINDOW, f"list_{b}"]
    d["unreported_derived"] = d.list_start + d.starts - d.completed - d.list_end
    d["identity_residual"] = d.unreported_published - d.unreported_derived
    d["completed_subtotal_residual"] = d.completed - (d.admitted + d.nonadmitted)
    d["removals_residual"] = o.loc[WINDOW, f"removals_{b}"] - (d.completed + d.unreported_published)
    d["share_published"] = o.loc[WINDOW, f"share_{b}"]
    d["share_rebuilt"] = d.unreported_published / (d.completed + d.unreported_published)
    d["share_residual"] = d.share_published - d.share_rebuilt
    d["net_change"] = d.list_end - d.list_start
    return d

TOL = 1e-6        # float tolerance: the headline flows are fractional, so I compare to a millionth of a pathway
head = identity("headline")
bad = head[head.identity_residual.abs() > TOL]
check("F1", f"headline identity closes, every month {mon_yy(FIRST_MONTH)}-{L}", bad.empty,
      "all months close" if bad.empty else f"fails in {list(bad.index)}: {bad.identity_residual.round(3).to_dict()}")
for c, label in [("completed_subtotal_residual", "completed = admitted + non-admitted"),
                 ("removals_residual", "total removals = completed + unreported"),
                 ("share_residual", "share = unreported / total removals")]:
    worst = head[c].abs().max()
    check("F1", f"headline: {label}", worst < TOL, f"max |residual| {worst:.2e}")
rebuilt = headline.loc[WINDOW, "total_est"]
f2 = (head.list_end - rebuilt).abs().max()
check("F2", "headline end-of-month list = main gate's rebuilt headline", f2 < TOL, f"max |diff| {f2:.2e}")
head[["list_start", "starts", "completed", "unreported_published", "unreported_derived", "identity_residual", "list_end"]].round(2)

,list_start,starts,completed,unreported_published,unreported_derived,identity_residual,list_end
2025-04,"7,418,598.00","1,707,444.00","1,450,094.00","286,883.00","286,883.00",0.00,"7,389,065.00"
2025-05,"7,389,065.00","1,757,217.00","1,500,695.00","289,637.00","289,637.00",0.00,"7,355,950.00"
2025-06,"7,355,950.00","1,828,309.00","1,558,632.00","257,791.00","257,791.00",0.00,"7,367,836.00"
2025-07,"7,367,836.00","1,942,537.33","1,648,944.52","271,149.81","271,149.81",0.00,"7,390,279.00"
2025-08,"7,390,279.00","1,626,863.33","1,404,088.24","209,442.10","209,442.10",-0.00,"7,403,612.00"
2025-09,"7,403,612.00","1,845,641.67","1,617,382.76","243,721.90","243,721.90",0.00,"7,388,149.00"
2025-10,"7,388,149.00","1,930,201.33","1,677,868.52","245,036.81","245,036.81",0.00,"7,395,445.00"
2025-11,"7,395,445.00","1,710,325.42","1,536,543.11","260,299.31","260,299.31",0.00,"7,308,928.00"
2025-12,"7,308,928.00","1,647,000.00","1,426,850.00","238,198.00","238,198.00",0.00,"7,290,880.00"
2026-01,"7,290,880.00","1,793,680.00","1,569,063.00","268,283.00","268,283.00",0.00,"7,247,214.00"


### Why the headline flows are not whole numbers

The press notice states NHS England's method for a trust that didn't submit:
its incomplete pathways are carried forward from its last submission (whole
numbers), but its completed pathways and clock starts are estimated as its
**per-working-day rate** in the month before the gap, multiplied by the month's
working days. That gives fractions. I test the explanation rather than assume
it: the fractional parts should appear only in the "with estimates" flow
columns, only in months with at least one estimated trust, and never in the
list itself.

In [5]:
adj = pd.read_excel(bronze_file(f"nhse_rtt_published_{REPORT_MONTH}/Missing-data-adjustments-to-{L}-*.xlsx"), sheet_name="Adjustments")
adj["period"] = adj["Estimated month"].dt.strftime("%Y-%m")
est_months = set(adj.loc[adj["RTT Type"].eq("Incomplete") & adj.period.isin(WINDOW), "period"])
frac = pd.DataFrame({c: (ov.loc[WINDOW, c] % 1).abs() > TOL for c in ov.columns if not c.startswith("share")})
frac_months = set(frac.index[frac.any(axis=1)])
check("F1", "fractions occur only in months with estimated trusts", frac_months <= est_months,
      f"{len(frac_months)} fractional months, {len(est_months)} months with estimates")
check("F1", "fractions occur only in 'with estimates' flow columns, never in the list",
      not frac[[c for c in frac.columns if c.endswith("_reported") or c.startswith("list_")]].any().any())
print("months with any fractional headline flow:", sorted(frac_months))

months with any fractional headline flow: ['2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09']


**Rounding rule for quoting.** The fractions are an artefact of NHS England's
estimation method, not precision. So in prose I quote monthly flows **to the
nearest thousand** ("about 352,000"), shares of exits **to one decimal place**,
and say "estimated" whenever a headline flow includes estimated trusts. Gold
keeps the unrounded figures, so anyone can check the rounding.

## Step 3 — Gate F3: the same identity, reported providers only

In [6]:
rep = identity("reported")
bad = rep[rep.identity_residual.abs() > TOL]
check("F3", f"reported identity closes, every month {mon_yy(FIRST_MONTH)}-{L}", bad.empty,
      "all months close" if bad.empty else f"fails in {list(bad.index)}")
for c, label in [("completed_subtotal_residual", "completed = admitted + non-admitted"),
                 ("removals_residual", "total removals = completed + unreported")]:
    check("F3", f"reported: {label}", rep[c].abs().max() < TOL, f"max |residual| {rep[c].abs().max():.2e}")
check("F3", "reported flows are whole numbers", not (rep[["starts", "completed", "unreported_published"]] % 1).any().any())
rep[["list_start", "starts", "completed", "unreported_published", "list_end"]].round(0)

,list_start,starts,completed,unreported_published,list_end
2025-04,"7,418,598.00","1,707,444.00","1,450,094.00","286,883.00","7,389,065.00"
2025-05,"7,389,065.00","1,757,217.00","1,500,695.00","289,637.00","7,355,950.00"
2025-06,"7,355,950.00","1,828,309.00","1,558,632.00","257,791.00","7,367,836.00"
2025-07,"7,367,836.00","1,913,956.00","1,626,548.00","356,221.00","7,299,023.00"
2025-08,"7,299,023.00","1,602,010.00","1,384,613.00","204,064.00","7,312,356.00"
2025-09,"7,312,356.00","1,818,303.00","1,595,960.00","237,806.00","7,296,893.00"
2025-10,"7,296,893.00","1,901,620.00","1,655,472.00","238,852.00","7,304,189.00"
2025-11,"7,304,189.00","1,669,006.00","1,502,727.00","311,458.00","7,159,010.00"
2025-12,"7,159,010.00","1,620,904.00","1,406,401.00","173,889.00","7,199,624.00"
2026-01,"7,199,624.00","1,767,584.00","1,548,614.00","262,636.00","7,155,958.00"


## Step 4 — Provider flows from the full CSV extracts (Silver for this notebook)

Notebook 02 keeps only the incomplete-pathway parts, and I'm not changing it.
So I read the other three parts here, from the same Bronze zips and with the
same rules: treatment function `C_999`, commissioner `NONC` excluded, and
"Total All", which includes completed pathways with an unknown clock start
(Overview note 7). Provider-level unreported removals then follow from

> unreported(p, t) = incomplete(p, t−1) + starts(p, t) − completed(p, t) − incomplete(p, t)

The first window month needs the previous month's provider lists, which this
Bronze doesn't hold (it starts in April 2025). So provider-level figures run
from the second month of the window.

In [7]:
PARTS = {"New RTT Periods - All Patients": "starts", "Completed Pathways For Admitted Patients": "admitted",
         "Completed Pathways For Non-Admitted Patients": "nonadmitted"}
rows = []
for period in WINDOW:
    zp = next((BRONZE_DIR / f"nhse_rtt_{period}").glob("*.zip"))
    with zipfile.ZipFile(zp) as z:
        with z.open([n for n in z.namelist() if n.endswith(".csv")][0]) as f:
            d = pd.read_csv(f, low_memory=False, usecols=["Provider Org Code", "Commissioner Org Code", "RTT Part Description",
                                                         "Treatment Function Code", "Total All"])
    d = d[d["Treatment Function Code"].eq("C_999") & d["RTT Part Description"].isin(PARTS)
          & ~d["Commissioner Org Code"].astype(str).str.startswith(RTT_UNIDENTIFIED_COMMISSIONER)]
    p = d.pivot_table(index="Provider Org Code", columns="RTT Part Description", values="Total All", aggfunc="sum").rename(columns=PARTS)
    p = p.reindex(columns=list(PARTS.values())).fillna(0)
    p.insert(0, "period", period)
    rows.append(p.reset_index().rename(columns={"Provider Org Code": "provider_code"}))
pflows = pd.concat(rows, ignore_index=True)
inc = pd.read_parquet(SILVER_DIR / "rtt" / "rtt_provider.parquet").query("part == 'incomplete'")[["period", "provider_code", "provider_name", "total"]]
pflows = pflows.merge(inc.rename(columns={"total": "incomplete"}), on=["period", "provider_code"], how="outer")
pflows[["starts", "admitted", "nonadmitted", "incomplete"]] = pflows[["starts", "admitted", "nonadmitted", "incomplete"]].fillna(0)
pflows["completed"] = pflows.admitted + pflows.nonadmitted
pflows.to_parquet(SILVER_DIR / "rtt" / "rtt_provider_flows.parquet", index=False)

# Gate F4: the providers must add up to NHS England's reported national flows, every month.
nat = pflows.groupby("period")[["starts", "admitted", "nonadmitted", "incomplete"]].sum()
for c, oc in [("starts", "starts_reported"), ("admitted", "admitted_reported"), ("nonadmitted", "nonadmitted_reported"),
              ("incomplete", "list_reported")]:
    diff = (nat[c] - ov.loc[WINDOW, oc]).abs()
    check("F4", f"provider {c} sum = NHS England's reported national figure, every month", (diff < TOL).all(),
          "all months exact" if (diff < TOL).all() else f"differs in {list(diff[diff >= TOL].index)}")
print(f"provider flow rows: {len(pflows):,}; providers: {pflows.provider_code.nunique()}")

provider flow rows: 9,088; providers: 557


## Step 5 — Gate F5: provider-level unreported removals, and what they can mean

I treat a merged trust as one entity in every month (North Bristol, RVJ, folded
into Bristol, RA7, as in notebook 05's panel). Otherwise the merger month would
show RVJ's whole list as a "removal" and RA7's as an "addition".

A provider-month falls into one of three kinds:

* **continuing**: the provider reported a list in both t−1 and t. Its unreported removals are interpretable, with caveats.
* **stopped submitting**: a list in t−1 and none in t. The identity books its whole previous list as a removal.
* **started or resumed**: no list in t−1, a list in t. The identity books its whole list as a negative removal.

The last two are submission artefacts, not patients leaving or joining a
list. They are exactly what NHS England's estimates exist to neutralise at
national level. They **can't** be neutralised per trust, because the estimates
aren't published at trust level for flows and can't be allocated to a trust that
didn't submit. So only "continuing" provider-months carry meaning.

In [8]:
ent = pflows.assign(entity=pflows.provider_code.replace({o: n for o, n, _ in TRUST_SUCCESSIONS}))
ent = ent.groupby(["entity", "period"])[["starts", "completed", "incomplete"]].sum().reset_index()
wide = ent.pivot_table(index="entity", columns="period", values=["starts", "completed", "incomplete"], aggfunc="sum").fillna(0)
prov_rows = []
for t_prev, t in zip(WINDOW[:-1], WINDOW[1:]):
    li_prev, li, st, co = wide[("incomplete", t_prev)], wide[("incomplete", t)], wide[("starts", t)], wide[("completed", t)]
    un = li_prev + st - co - li
    kind = np.select([(li_prev > 0) & (li > 0), (li_prev > 0) & (li == 0), (li_prev == 0) & (li > 0)],
                     ["continuing", "stopped submitting", "started or resumed"], "no list either month")
    prov_rows.append(pd.DataFrame({"period": t, "entity": wide.index, "list_start": li_prev.values, "starts": st.values,
                                   "completed": co.values, "list_end": li.values, "unreported": un.values, "kind": kind}))
prov = pd.concat(prov_rows, ignore_index=True)
names = pflows.dropna(subset=["provider_name"]).drop_duplicates("provider_code", keep="last").set_index("provider_code").provider_name
prov["name"] = prov.entity.map(names)

# F5: the providers' unreported removals must sum to the national reported figure.
sums = prov.groupby("period").unreported.sum()
d5 = (sums - rep.loc[WINDOW[1:], "unreported_published"]).abs()
check("F5", f"provider unreported removals sum to the national reported figure, {mon_yy(WINDOW[1])}-{L}",
      (d5 < TOL).all(), "all months exact" if (d5 < TOL).all() else f"differs in {list(d5[d5 >= TOL].index)}")

split = prov.pivot_table(index="period", columns="kind", values="unreported", aggfunc="sum").fillna(0)
split["national_reported"] = rep.loc[WINDOW[1:], "unreported_published"]
split["national_headline"] = head.loc[WINDOW[1:], "unreported_published"]
prov.to_csv(OUT / "provider_unreported.csv", index=False)
split.to_csv(OUT / "unreported_by_kind.csv")
split.round(0)

kind,continuing,no list either month,started or resumed,stopped submitting,national_reported,national_headline
period,,,,,,
2025-05,"289,165.00",0.00,-14.00,486.00,"289,637.00","289,637.00"
2025-06,"263,403.00",2.00,"-5,614.00",0.00,"257,791.00","257,791.00"
2025-07,"265,863.00",-5.00,-894.00,"91,257.00","356,221.00","271,150.00"
2025-08,"201,210.00",-1.00,-53.00,"2,908.00","204,064.00","209,442.00"
2025-09,"235,962.00","2,223.00",-383.00,4.00,"237,806.00","243,722.00"
2025-10,"239,750.00",71.00,"-1,207.00",238.00,"238,852.00","245,037.00"
2025-11,"253,647.00",-26.00,"-2,880.00","60,717.00","311,458.00","260,299.00"
2025-12,"227,078.00",-68.00,"-54,406.00","1,285.00","173,889.00","238,198.00"
2026-01,"262,803.00",-122.00,-45.00,0.00,"262,636.00","268,283.00"


**Per-provider caveats** (from NHS England's explainer):

* transfers between providers move a pathway onto one list without a clock start and off another without a completion, so at trust level they create offsetting unreported additions and removals that cancel nationally;
* clock nullifications, duplicates found in validation, completions recorded after the deadline and incorrectly reported clock starts all land here too;
* a trust that didn't submit has no provider figure at all; NHS England's estimates exist only in the national series.

## Step 6 — Write Gold and close the gate

In [9]:
flows = pd.concat({"headline": head, "reported": rep}, names=["basis", "period"]).reset_index()
flows.to_csv(OUT / "flows_national.csv", index=False)
res = pd.DataFrame(checks)
res.to_csv(OUT / "flows_gate.csv", index=False)
print(res.to_string(index=False))
failed = res[~res["pass"]]
if len(failed):
    raise SystemExit(f"FLOWS GATE: FAIL ({len(failed)} of {len(res)} checks)")
print(f"\nFLOWS GATE: PASS ({len(res)} of {len(res)} checks)")

gate                                                                          check  pass                                           note
  F1                              headline identity closes, every month Apr25-Sep26  True                               all months close
  F1                                  headline: completed = admitted + non-admitted  True                        max |residual| 0.00e+00
  F1                              headline: total removals = completed + unreported  True                        max |residual| 0.00e+00
  F1                                  headline: share = unreported / total removals  True                        max |residual| 0.00e+00
  F2                      headline end-of-month list = main gate's rebuilt headline  True                            max |diff| 0.00e+00
  F1                           fractions occur only in months with estimated trusts  True 13 fractional months, 15 months with estimates
  F1       fractions occur only in 'with 